# RAG with Local Documents or Google Drive

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) pipeline:

1. Load PDF and text files either from a local folder or directly from Google Drive.
2. Chunk documents and generate embeddings using `nomic-embed-text-v1.5` or `all-MiniLM-L6-v2`.
3. Store embeddings in ChromaDB.
4. On a user query, retrieve relevant chunks and rerank with a cross-encoder.
5. Provide a Gradio interface where the user selects an LLM (OpenAI or local Ollama) and asks a question.
6. The chosen model answers using the retrieved context.

You can choose your document source at runtime: **local folder** or **Google Drive** (via API).


In [ ]:
## Install Dependencies

!uv pip install -q chromadb gradio openai ollama sentence-transformers pypdf google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [ ]:
import os
import getpass
from gdrive_utils import authenticate, list_files, download_file
from openai import OpenAI
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel


## Google Drive API Setup (Required only for remote access)

1. Go to the [Google Cloud Console](https://console.cloud.google.com/).
2. Create a new project (or select an existing one).
3. Enable the **Google Drive API**.
4. Go to **Credentials** → **+ Create Credentials** → **OAuth client ID**.
5. Choose **Desktop app** as the application type.
6. Download the JSON file and rename it to `credentials.json`. Place it in the same folder as this notebook.
7. The notebook uses `gdrive_utils.py` module that handles authentication and downloading with skip logic.

> **Note:** On first run, a browser window will open for you to log in and grant permissions. A `token.pickle` file will be saved for subsequent runs.

In [ ]:
# Change the source to 'local' or 'gdrive' to switch between local and Google Drive
source = 'local'

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200 

EMBEDDING_MODEL = "all-MiniLM-L6-v2" # or "nomic-embed-text-v1.5"
EMBEDDING_HF_NAME = "nomic-ai/nomic-embed-text-v1.5"
CHROMA_DIR = "./chroma_db"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_RETRIEVAL = 15
TOP_K_RERANK = 10

if source == 'gdrive':
    # Check for credentials file
    if not os.path.exists('credentials.json'):
        raise FileNotFoundError("credentials.json missing. Follow Google Drive API setup.")

    print("Authenticating with Google Drive...")
    service = authenticate()

    folder_id = input("Enter the Google Drive folder ID: ").strip()
    print("Scanning folder...")
    files = list_files(service, folder_id)
    print(f"Found {len(files)} PDF/text files.")

    # Download to local folder
    DOCUMENTS_FOLDER = "./gdrive_docs"
    print("Downloading new files...")
    for f in files:
        download_file(service, f['id'], f['name'], DOCUMENTS_FOLDER)
    print("Download complete.")

elif source == "local":
    DOCUMENTS_FOLDER = './gdrive_docs'
    if not os.path.isdir(DOCUMENTS_FOLDER):
        raise FileNotFoundError(f"Folder not found: {DOCUMENTS_FOLDER}")
else:
    raise ValueError("Invalid source. Choose 'local' or 'gdrive'.")

# OpenAI API key (optional)
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY and OPENAI_API_KEY.startswith('sk-'):
    import openai
    openai.api_key = OPENAI_API_KEY
    print("OpenAI API key loaded successfully.")
else:
    print("No OpenAI API key provided. Some features may be limited.")

local_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
remote_client = OpenAI()

print(f"\nDocuments will be read from: {DOCUMENTS_FOLDER}")

### Load Documents Function

In [ ]:
def load_documents(folder_path):
    texts = []
    metadata = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith(".txt"):
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()
                texts.append(text)
                metadata.append({"source": file_path, "type": "txt"})
            elif file.endswith(".pdf"):
                try:
                    reader = PdfReader(file_path)
                    text = ""
                    for page in reader.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + "\n"
                    if text.strip():
                        texts.append(text)
                        metadata.append({"source": file_path, "type": "pdf"})
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
    return texts, metadata

print("Loading documents...")
doc_texts, doc_metadata = load_documents(DOCUMENTS_FOLDER)
print(f"Loaded {len(doc_texts)} documents.")

### Chunk Documents

In [ ]:
documents = []
for text, meta in zip(doc_texts, doc_metadata):
    documents.append({
        "page_content": text,
        "metadata": meta
    })

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = []
chunk_metadata = []
for i, (text, meta) in enumerate(zip(doc_texts, doc_metadata)):
    doc_chunks = text_splitter.split_text(text)
    for j, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        chunk_metadata.append({
            "doc_index": i,
            "chunk_index": j,
            "source": meta["source"],
            "type": meta["type"]
        })

print(f"Created {len(chunks)} chunks.")

### Embedding Function

In [ ]:
# Load embedding model
if EMBEDDING_MODEL == "nomic-embed-text-v1.5":
    model_name = "nomic-ai/nomic-embed-text-v1.5"
else:
    model_name = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(model_name)

# Custom embedding function for Chroma
class SentenceTransformerEmbeddingFunction(embedding_functions.EmbeddingFunction):
    def __init__(self, model):
        self.model = model

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        embeddings = self.model.encode(texts, convert_to_tensor=False).tolist()
        return embeddings

chroma_ef = SentenceTransformerEmbeddingFunction(embedding_model)

### Create Chroma Collection and Add Chunks

In [ ]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

collection_name = "rag_docs"
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(
    name=collection_name,
    embedding_function=chroma_ef,
    metadata={"hnsw:space": "cosine"}
)

# Add chunks in batches
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch_chunks = chunks[i:i+BATCH_SIZE]
    batch_metadata = chunk_metadata[i:i+BATCH_SIZE]
    ids = [f"chunk_{i+j}" for j in range(len(batch_chunks))]
    collection.add(
        documents=batch_chunks,
        metadatas=batch_metadata,
        ids=ids
    )

print(f"Added {len(chunks)} chunks to Chroma collection.")

### Reranker Setup

In [ ]:
class Result(BaseModel):
    metadata: dict
    page_content: str

def embed_question(question):
    return embedding_model.encode(question, convert_to_tensor=False).tolist()

def fetch_context(query):
    question_embedding = embed_question(query)
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=TOP_K_RETRIEVAL
    )
    chunks = []
    for result in zip(results['documents'][0], results['metadatas'][0]):
        chunks.append(Result(metadata=result[1], page_content=result[0]))
    return chunks

In [ ]:
reranker = CrossEncoder(RERANKER_MODEL)

### Retrieve and Rerank Function

In [ ]:
def retrieve_and_rerank(query):
    results = collection.query(
        query_texts=[query],
        n_results=TOP_K_RETRIEVAL
    )
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]

    if not documents:
        return [], []

    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)

    scored = list(zip(scores, documents, metadatas))
    scored.sort(reverse=True, key=lambda x: x[0])

    top_scored = scored[:TOP_K_RERANK]
    top_docs = [doc for _, doc, _ in top_scored]
    top_metas = [meta for _, _, meta in top_scored]
    return top_docs, top_metas

### Build Prompt with Context

In [ ]:
system_prompt = '''
You are a helpful assistant.
If the provided context is relevant to the question, make use of it to answer the question.
Make sure not to mention the context in your answer, just answer the question based on the context if it makes the answer more accurate.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
'''

def build_prompt(query, context_chunks):
    context = "\n\n---\n\n".join(context_chunks)
    prompt = f"""
    You are a helpful assistant. 
    Use the following pieces of context to answer the question at the end.
    Only use the context if it makes the answer more accurate.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

### Model Interfaces

In [ ]:
def call_openai(model_name, prompt):
    try:
        print(f"Streaming response from {model_name}...")
        response = openai.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
                ],
            temperature=0.7,
            stream=True
        )
        result = ''
        for chunk in response:
            result += chunk.choices[0].delta.content or ''
            yield result
    except Exception as e:
        print(f"Error calling OpenAI: {e}")
        yield ''

def call_ollama(model_name, prompt):
    try:
        print(f"Streaming response from {model_name}...")
        response = local_client.chat.completions.create(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ],
            temperature=0.7,
            stream=True
        )
        result = ''
        for chunk in response:
            result += chunk.choices[0].delta.content or ''
            yield result
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        yield ''

In [ ]:
def get_local_models():
        try:
            response = local_client.models.list()
            return [model.id for model in response.data]
        except Exception as e:
            print(f'Error listing models: {e}')
            return []

Gradio Interface

In [ ]:
import gradio as gr

# Available models
openai_models = ['gpt-4o-mini', 'gpt-5-nano', 'gpt-4.1-mini']
ollama_models = get_local_models()

model_choices = {"OpenAI": openai_models, "Ollama": ollama_models}

In [ ]:
def answer_question(provider, model, question):
    if not question.strip():
        return "Please enter a question."

    top_chunks, top_metas = retrieve_and_rerank(question)

    if not top_chunks:
        print("No relevant documents found.")

    sources = "\n\nSources:\n" + "\n".join([f'{m["source"]}, chunk {m["chunk_index"]}' for m in top_metas])
    print(f'Using the following sources to answer the question: {sources}')

    prompt = build_prompt(question, top_chunks)

    if provider == "OpenAI":
        yield from call_openai(model, prompt)
    else:
        yield from call_ollama(model, prompt)

    # sources = "\n\nSources:\n" + "\n".join([m["source"] for m in top_metas])
    # return answer 

In [ ]:
with gr.Blocks(title="RAG with Local or Google Drive Documents") as demo:
    gr.Markdown("# RAG with your documents")
    gr.Markdown("Select a model, ask a question, and get answers based on your indexed PDFs/text files.")

    with gr.Row():
        provider_dropdown = gr.Dropdown(choices=list(model_choices.keys()), label="Provider", value="OpenAI")
        model_dropdown = gr.Dropdown(choices=openai_models, label="Model", value="gpt-4o-mini")

    question_input = gr.Textbox(label="Your Question", lines=3)
    answer_output = gr.Textbox(label="Answer", lines=10, interactive=False)

    submit_btn = gr.Button("Get Answer")

    def update_models(provider):
        return gr.Dropdown(choices=model_choices[provider])

    provider_dropdown.change(update_models, inputs=provider_dropdown, outputs=model_dropdown)

    submit_btn.click(answer_question, inputs=[provider_dropdown, model_dropdown, question_input], outputs=answer_output)

Launch the Interface

In [ ]:
demo.launch(inbrowser=True)